# Join Algorithms, Their Crossover, and Why the Order Matters More

Companion notebook for the [Join Algorithms and Join Ordering](https://codemia.ai/courses/database_internals/dbi_join_algorithms_and_join_ordering) lesson on Codemia.

**What you will build**: nested loop, hash and sort merge joins that all produce the same answer,
a page based cost model that says which one an engine should pick, and a join order enumerator over
four tables. By the end you will be able to say where each algorithm wins, how much a bad join order
costs compared to a bad algorithm choice, and which of the two a planner gets wrong more often.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import itertools
import time
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(29)

OUTER = 2000      # rows on the probing side for the measured runs
KEYS = 4000       # distinct join key values
PAGE_ROWS = 100   # rows per page, for the cost model
MEMORY = 200      # pages of working memory available to a join

## Concept recap

Every join algorithm in common use is one of three ideas.

A nested loop join walks the outer input and, for each row, looks for matches in the inner one. With
no index on the inner side that is a scan per outer row, which is quadratic and only acceptable when
one side is tiny. With an index on the join column it is a descent per outer row, which is the
fastest plan in existence when the outer side is small and the worst one when it is not.

A hash join builds a hash table on the smaller input and probes it once per row of the larger one.
It is linear in both inputs, it needs the build side to fit in memory or it has to partition both
sides to disk first, and it only works for equality predicates.

A sort merge join sorts both inputs on the join column and walks them together. It is dominated by
the sort, so it is attractive when an input arrives sorted already, for instance from an index scan,
and it handles range predicates and produces sorted output, which a later operator may want.

The planner chooses between them from estimated cardinalities, which means everything in the
statistics notebook applies here. And as the last section shows, the choice of algorithm is usually
the smaller of the two decisions.

## Three joins that return the same rows

In [ ]:
def make(n, keys=KEYS, seed=0):
    gen = np.random.default_rng(seed)
    return [(int(k), i) for i, k in enumerate(gen.integers(0, keys, n))]


def nested_loop(outer, inner):
    """No index on the inner side: scan it once per outer row."""
    out, compares = [], 0
    for ko, vo in outer:
        for ki, vi in inner:
            compares += 1
            if ko == ki:
                out.append((ko, vo, vi))
    return out, compares


def index_nested_loop(outer, index):
    """An index already exists on the inner join column, so no build cost is charged."""
    out, probes = [], 0
    for ko, vo in outer:
        probes += 1
        for vi in index.get(ko, ()):
            out.append((ko, vo, vi))
    return out, probes


def hash_join(outer, inner):
    """Build on the smaller side, probe with the larger."""
    build, probe = (inner, outer) if len(inner) <= len(outer) else (outer, inner)
    table = defaultdict(list)
    for k, v in build:
        table[k].append(v)
    out, touched = [], len(build)
    for k, v in probe:
        touched += 1
        for other in table.get(k, ()):
            out.append((k, v, other) if build is inner else (k, other, v))
    return out, touched


def sort_merge(outer, inner):
    a, b = sorted(outer), sorted(inner)
    out, i, j, steps = [], 0, 0, 0
    while i < len(a) and j < len(b):
        steps += 1
        if a[i][0] < b[j][0]:
            i += 1
        elif a[i][0] > b[j][0]:
            j += 1
        else:
            ki = i
            while ki < len(a) and a[ki][0] == b[j][0]:
                kj = j
                while kj < len(b) and b[kj][0] == a[ki][0]:
                    out.append((a[ki][0], a[ki][1], b[kj][1]))
                    kj += 1
                ki += 1
            i, j = ki, kj
    return out, steps


outer = make(OUTER, seed=1)
inner = make(OUTER, seed=2)
index = defaultdict(list)
for k, v in inner:
    index[k].append(v)

counts = {
    "nested loop": len(nested_loop(outer, inner)[0]),
    "index nested loop": len(index_nested_loop(outer, index)[0]),
    "hash": len(hash_join(outer, inner)[0]),
    "sort merge": len(sort_merge(outer, inner)[0]),
}
print(counts)
assert len(set(counts.values())) == 1, "the four joins must agree"
print("all four agree on %d result rows" % counts["hash"])

## Where each one wins, measured

In [ ]:
sizes = [10, 50, 200, 1000, 2000, 5000]
timings = {k: [] for k in ("nested loop", "index nested loop", "hash", "sort merge")}

for m in sizes:
    inner_m = make(m, seed=3)
    idx = defaultdict(list)
    for k, v in inner_m:
        idx[k].append(v)

    t = time.perf_counter(); nested_loop(outer, inner_m); timings["nested loop"].append(time.perf_counter() - t)
    t = time.perf_counter(); index_nested_loop(outer, idx); timings["index nested loop"].append(time.perf_counter() - t)
    t = time.perf_counter(); hash_join(outer, inner_m); timings["hash"].append(time.perf_counter() - t)
    t = time.perf_counter(); sort_merge(outer, inner_m); timings["sort merge"].append(time.perf_counter() - t)

print("%12s %14s %20s %12s %12s" % ("inner rows", "nested loop", "index nested loop", "hash", "sort merge"))
for i, m in enumerate(sizes):
    print("%12d %13.1fms %19.1fms %11.1fms %11.1fms" %
          (m, 1000 * timings["nested loop"][i], 1000 * timings["index nested loop"][i],
           1000 * timings["hash"][i], 1000 * timings["sort merge"][i]))

In [ ]:
plt.figure(figsize=(9, 4.2))
styles = {"nested loop": "#d1495b", "index nested loop": "#5b8def",
          "hash": "#2f8f6b", "sort merge": "#8a5bd6"}
for name, color in styles.items():
    plt.plot(sizes, [1000 * t for t in timings[name]], "o-", color=color, lw=2, label=name)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("rows on the inner side (outer side fixed at %d)" % OUTER)
plt.ylabel("milliseconds")
plt.title("the same join, four ways")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

The nested loop is the only line that rises with the inner size, and it rises linearly in it, which
with a fixed outer side is what the quadratic term looks like from here. At ten inner rows it is a
few tenths of a millisecond behind the others and at five thousand it is three hundred times behind,
which is why a planner that underestimates the inner side can produce a query that never finishes.

The index nested loop is nearly flat, because the work is one descent per outer row and barely
depends on how large the inner table is. It is the fastest of the four at every size here, and it is
charged no build cost because the index already exists. That caveat is the entire story: the plan is
free only because somebody created the index in advance.

Hash and sort merge both scale with the sum of the inputs rather than the product. Hash wins at
these sizes because sorting costs more than hashing when neither input arrives sorted, which is the
usual case.

## What the planner actually compares

In [ ]:
def pages(rows):
    return max(1, rows // PAGE_ROWS)

def cost_nested_loop(m, n):
    """Block nested loop: read the outer once, the inner once per block of memory."""
    return pages(m) + pages(m) / max(1, MEMORY - 2) * pages(n)

def cost_index_nested_loop(m, height=3):
    return pages(m) + m * height          # one descent per outer row

def cost_hash(m, n):
    build, probe = min(m, n), max(m, n)
    if pages(build) <= MEMORY:
        return pages(m) + pages(n)        # one pass each
    return 3 * (pages(m) + pages(n))      # partition out and back

def cost_sort_merge(m, n):
    def sort_cost(r):
        p = pages(r)
        passes = max(1, int(np.ceil(np.log(max(p / MEMORY, 1.0001)) / np.log(MEMORY))) + 1)
        return 2 * p * passes
    return sort_cost(m) + sort_cost(n) + pages(m) + pages(n)

print("%12s %14s %18s %10s %12s %14s" %
      ("inner rows", "block nested", "index nested", "hash", "sort merge", "cheapest"))
for n in (100, 1000, 10000, 100000, 1000000):
    row = {
        "block nested": cost_nested_loop(1000000, n),
        "index nested": cost_index_nested_loop(1000000),
        "hash": cost_hash(1000000, n),
        "sort merge": cost_sort_merge(1000000, n),
    }
    best = min(row, key=row.get)
    print("%12d %14.0f %18.0f %10.0f %12.0f %14s" %
          (n, row["block nested"], row["index nested"], row["hash"], row["sort merge"], best))

This is the comparison in the units an engine uses, pages rather than seconds, with a million row
outer side.

Two things are different from the measured plot. The index nested loop is no longer free, because
with a million outer rows a descent each is a million random page accesses and the model charges for
them; it is the right plan for a small outer side and the wrong one here. And hash wins across the
range, with sort merge close behind once both inputs are large enough that the sort dominates
nothing in particular.

The block nested loop is also better than its reputation at small inner sizes, because reading a
hundred page table once per block of memory is cheap. That is why engines still choose it, and why
it appears in plans where the inner side is a tiny lookup table.

## The decision that costs more than the algorithm

In [ ]:
# A four table query: one large fact table and three dimensions.
SIZES = {"orders": 1000000, "customers": 50000, "regions": 50, "promos": 200}
DISTINCT = {("orders", "customers"): 50000, ("orders", "regions"): 50,
            ("orders", "promos"): 200, ("customers", "regions"): 50}

def selectivity(a, b):
    key = (a, b) if (a, b) in DISTINCT else (b, a)
    return 1.0 / DISTINCT[key] if key in DISTINCT else None   # None means no predicate

def order_cost(order, sizes):
    """Left deep plan: join tables in this order, charging the size of each intermediate."""
    joined = [order[0]]
    card = sizes[order[0]]
    total = 0.0
    for nxt in order[1:]:
        sel = None
        for t in joined:
            s = selectivity(t, nxt)
            if s is not None:
                sel = s if sel is None else sel * s
        card = card * sizes[nxt] * (sel if sel is not None else 1.0)
        total += card
        joined.append(nxt)
    return total

orders = list(itertools.permutations(SIZES))
costed = sorted(((order_cost(o, SIZES), o) for o in orders))
print("%-56s %16s" % ("join order", "rows through the plan"))
for cost, o in costed[:3]:
    print("%-56s %16.0f" % (" then ".join(o), cost))
print("...")
for cost, o in costed[-2:]:
    print("%-56s %16.0f" % (" then ".join(o), cost))
print()
print("best %.0f, worst %.0f, a factor of %.0f between them"
      % (costed[0][0], costed[-1][0], costed[-1][0] / costed[0][0]))

Twenty four orders for four tables, and the gap between the best and the worst is far larger than
the gap between any two algorithms in the previous section. The worst orders are the ones that join
two tables with no predicate between them, producing a cross product that every later step has to
carry.

The number of orders is the reason planning is hard. Four tables give twenty four left deep orders,
eight tables give forty thousand, twelve tables give half a billion, and that is before considering
bushy plans or which algorithm to use at each step. This is why planners use dynamic programming up
to some number of tables and switch to a greedy search beyond it, and why a join of fifteen tables
can get a plan that is merely acceptable.

In [ ]:
# Now give the planner one wrong cardinality: it believes a filter cuts the fact
# table to a thousand rows, when the filter actually matches all of it.
BELIEVED = dict(SIZES, orders=1000)
believed_best = min(orders, key=lambda o: order_cost(o, BELIEVED))
true_best = costed[0][1]

print("planner picks   %s" % " then ".join(believed_best))
print("truly best is   %s" % " then ".join(true_best))
print()
print("%-24s %18s" % ("", "real rows through the plan"))
print("%-24s %18.0f" % ("the picked order", order_cost(believed_best, SIZES)))
print("%-24s %18.0f" % ("the best order", order_cost(true_best, SIZES)))
print("%-24s %17.1fx" % ("penalty",
      order_cost(believed_best, SIZES) / order_cost(true_best, SIZES)))

One wrong cardinality on the fact table, and the planner leads with it rather than building up from
the small dimensions, which pushes an order of magnitude more rows through the plan. The estimate was
wrong about one filter, every individual join is still a reasonable join, and the plan is ten times
worse.

The real penalty would be larger than this. The model here only counts rows flowing through the
plan; an engine that believes an input is a thousand rows will also choose a nested loop over it and
request a small amount of memory for its hash tables, so the same error degrades the algorithm
choice at every step of the order it picked.

Put the two sections together and the priority is clear. Choosing the wrong algorithm costs a factor
of a few, except for the nested loop case where it costs a factor of everything. Choosing the wrong
order can cost several orders of magnitude on a query with no obviously bad step in it. And both
decisions are made from the same estimates, so the way to improve them is not to tune the planner
but to give it statistics that describe the data, including the correlations.

## Variations: a skewed join key

In [ ]:
def skewed(n, hot_fraction, seed=7):
    gen = np.random.default_rng(seed)
    hot = gen.random(n) < hot_fraction
    keys = np.where(hot, 0, gen.integers(1, KEYS, n))
    return [(int(k), i) for i, k in enumerate(keys)]

print("%14s %14s %16s %14s" % ("hot fraction", "result rows", "hash build max", "hash time"))
for frac in (0.0, 0.05, 0.2, 0.5):
    o = skewed(OUTER, frac, seed=8)
    i = skewed(OUTER, frac, seed=9)
    t = time.perf_counter()
    res, _ = hash_join(o, i)
    elapsed = time.perf_counter() - t
    table = defaultdict(int)
    for k, _v in i:
        table[k] += 1
    print("%14.2f %14d %16d %13.1fms" % (frac, len(res), max(table.values()), 1000 * elapsed))

Skew in the join key does not slow the hash table down. The lookup is still constant time and the
bucket for the hot key is a list that is appended to and iterated once, and the measured time tracks
the number of rows emitted rather than the skew: a thousand times the output for about a hundred
times the time.

What it does is multiply the output. Five per cent of each side sharing one key means those rows
join to each other in every combination, and the result grows with the product of the two hot
counts. The join is not slow because the algorithm degraded, it is slow because it is correctly
producing an enormous number of rows.

That is the shape of most real join incidents. The plan is reasonable, the estimate assumed the keys
were spread evenly, and the output is a hundred times larger than anyone expected. The hot key is
usually a default, a placeholder, or a null that somebody replaced with zero.

## What to try next

- Make the hash build side larger than memory and implement the partitioning pass, then compare the
  measured cost against the three times model above.
- Allow bushy plans in the enumerator and count how many more orders there are, then check whether
  the best bushy plan beats the best left deep one on this query.
- Replace the exhaustive search with the greedy rule of always joining the pair with the smallest
  output next, and see how often it finds the best order on randomly generated queries.
- Add a sorted input to the sort merge join, charging no sort cost on that side, and find the point
  where it beats the hash join. That is the case an index scan feeding a merge join is built for.